In [1]:
import torch

print(torch.__version__)
import numpy as np
from numpy import pi
from scipy.spatial.transform import Rotation as R
import matplotlib.pyplot as plt

2.3.1+cu121


In [2]:
# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available. Number of CUDA devices:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
        print(f"  - Capability: {torch.cuda.get_device_capability(i)}")
        print(f"  - Memory Allocated: {torch.cuda.memory_allocated(i)} bytes")
        print(f"  - Memory Cached: {torch.cuda.memory_reserved(i)} bytes")
else:
    print("CUDA is not available.")

CUDA is available. Number of CUDA devices: 1
Device 0: NVIDIA GeForce RTX 4060 Laptop GPU
  - Capability: (8, 9)
  - Memory Allocated: 0 bytes
  - Memory Cached: 0 bytes


# Section 0: Proofs of concepts with quaternions and integration of rotations. 

In [3]:
# r = R.from_quat([0, 0, 1, 2])
# r_hat = r.as_euler('zyx', degrees=False)
# R.from_euler('zyx', r_hat, degrees=False).as_quat()

In [4]:
# def plot_rotated_axes(ax, r, name=None, offset=(0, 0, 0), scale=1):
#     colors = ("#FF6666", "#005533", "#1199EE")  # Colorblind-safe RGB
#     loc = np.array([offset, offset])
#     for i, (axis, c) in enumerate(zip((ax.xaxis, ax.yaxis, ax.zaxis),
#                                       colors)):
#         axlabel = axis.axis_name
#         axis.set_label_text(axlabel)
#         axis.label.set_color(c)
#         axis.line.set_color(c)
#         axis.set_tick_params(colors=c)
#         line = np.zeros((2, 3))
#         line[1, i] = scale
#         line_rot = r.apply(line)
#         line_plot = line_rot + loc
#         ax.plot(line_plot[:, 0], line_plot[:, 1], line_plot[:, 2], c)
#         text_loc = line[1]*1.2
#         text_loc_rot = r.apply(text_loc)
#         text_plot = text_loc_rot + loc[0]
#         ax.text(*text_plot, axlabel.upper(), color=c,
#                 va="center", ha="center")
#     ax.text(*offset, name, color="k", va="center", ha="center",
#             bbox={"fc": "w", "alpha": 0.8, "boxstyle": "circle"})

In [5]:
# r0 = R.identity()
# r1 = R.from_euler("ZYX", [90, -30, 0], degrees=True)  # intrinsic
# r2 = R.from_euler("zyx", [90, -30, 0], degrees=True)  # extrinsic
# 
# ax = plt.figure().add_subplot(projection="3d", proj_type="ortho")
# plot_rotated_axes(ax, r0, name="r0", offset=(0, 0, 0))
# plot_rotated_axes(ax, r1, name="r1", offset=(3, 0, 0))
# plot_rotated_axes(ax, r2, name="r2", offset=(6, 0, 0))
# _ = ax.annotate(
#     "r0: Identity Rotation\n"
#     "r1: Intrinsic Euler Rotation (ZYX)\n"
#     "r2: Extrinsic Euler Rotation (zyx)",
#     xy=(0.6, 0.7), xycoords="axes fraction", ha="left"
# )
# ax.set(xlim=(-1.25, 7.25), ylim=(-1.25, 1.25), zlim=(-1.25, 1.25))
# ax.set(xticks=range(-1, 8), yticks=[-1, 0, 1], zticks=[-1, 0, 1])
# ax.set_aspect("equal", adjustable="box")
# ax.figure.set_size_inches(6, 5)
# plt.tight_layout()

In [6]:
# r0 = R.from_quat([0, 0, 0, 1])
# r1_q = [1, 0, 0, 0]
# r2_q = [0, 1, 0, 0]
# r1 = R.from_quat(r1_q)
# r2 = R.from_quat(r2_q)
# 
# print((r0 * r1 * r2).as_quat())
# print((r2 * r1 * r0).as_quat())

In [7]:
import Rotations as Rot

In [8]:
# angular_displacements = torch.tensor([[0.1, 0.2, 0.3], [0.0, pi, 0.0]])  # Example tensor of angular displacements
# rotations = Rot.exp_quat(angular_displacements)
# 
# print(rotations)

In [9]:
# torch.device("cuda:0")

# Training Task 0.1q and 0.2q

## Generating datasets


In [10]:
from utils import goto_project_root
from utils.path_settings import MODEL_SAVE_PATH, DATA_PATH, LOG_PATH
import SimulateDatasets.GenTrainingData as g
from utils import create_splits, get_dataloaders
from Network_models import Trainer

configs = {
    "model_specs": {
        "model_name": "CustomRNN", # To diffrentiate from the several types of RNN cells.
        "model_path": "Network_models.RNN_models",
        "model_params": {
            "input_size": 4,
            "hidden_size": 64,
            "num_layers": 1,
            "output_size": 3,
            "cell_type": "GRU",
        }
    },

    "device": "cuda",
    "optimizer_specs": {
        "optimizer_name": "Adam",
        "optimizer_params": {
            "lr": 0.001
        }
    },

    "distance_loss": "geodesic",
    "regularisation_loss": "L2",
    "distance_weight": 0.5,

    "save_path": MODEL_SAVE_PATH + "RNN_model",
    "log_path": "runs",

    "training_config": {
        "task_id": "0.1q",
        "batch_size": 128,
        "seq_len": 1000,
        "prep_phase": 500,
    }
}

configs["training_config"]["save_path"] = DATA_PATH + ("RNN_model_" + configs["training_config"]["task_id"] + ".pth")
data1 = g.gen_training_data(configs['training_config'])
dataloaders = get_dataloaders(data1, batch_size=configs["training_config"]["batch_size"], k=5)
rnn_trainer = Trainer(configs)
for i, (train_loader, test_loader) in enumerate(dataloaders):
    print(f"Training on split {i+1}")
    rnn_trainer.train(train_loader, test_loader, epochs=10)


## improve transform speed

In [19]:
import torch
import torch.nn.functional as F
import pytorch3d.transforms as transforms

def integrate_velocities(omega, dt=1.0):
    """
    Integrate angular velocities to rotations using PyTorch operations.
    
    Args:
        omega (torch.Tensor): A tensor of shape (batch_size, seq_length, 3)
                              representing the angular velocity vector.
        dt (float): The time step.
    
    Returns:
        torch.Tensor: A tensor of shape (batch_size, 4) representing the integrated quaternion rotations.
    """
    assert omega.shape[-1] == 3
    
    # Initial quaternion (identity rotation)
    q0 = torch.tensor([0.0, 0.0, 0.0, 1.0], device=omega.device).unsqueeze(0).expand(omega.shape[0], -1)
    
    # Compute quaternion updates
    omega_norm = torch.norm(omega, dim=-1, keepdim=True)
    omega_hat = omega / omega_norm.clamp(min=1e-8)
    theta = omega_norm * dt
    q = torch.cat([torch.cos(theta / 2), torch.sin(theta / 2) * omega_hat], dim=-1)
    
    # Integrate quaternions
    integrated_q = transforms.integrate_quaternion(q, q0)
    
    return integrated_q.squeeze(1)  # Shape: (batch_size, 4)

# Example usage:
omega = torch.randn(100, 500, 3)  # Example angular velocity tensor
integrated_quaternions = integrate_velocities(omega, dt=1.0)
print(integrated_quaternions.shape)  # Output shape: torch.Size([100, 4])

ModuleNotFoundError: No module named 'pytorch3d'